## Reranking (Cross-Encoder / Cohere Rerank)

1차 벡터 검색은 **재현율(recall)** 을 위해 후보를 넓게 가져오고,  
리랭커는 그 후보를 **질문-문서 쌍**으로 다시 점수화해 **정밀도(precision)** 를 높인다.

| 구분 | Bi-Encoder (1차 검색) | Cross-Encoder / Cohere Rerank |
|---|---|---|
| 입력 | 질문·문서를 **각각** 임베딩 | 질문+문서를 **한 쌍**으로 입력 |
| 장점 | 빠르고 대규모 인덱스에 적합 | 관련성 판별이 더 정밀 |
| 단점 | 표현 간극·유사 키워드에 취약 | 후보 수만큼 비용·지연 증가 |
| 역할 | top-$k$ 후보 확보 | top-$k$ → top-$n$ 재정렬 |

```text
pip install langchain-classic langchain-community langchain-openai faiss-cpu
pip install sentence-transformers   # Cross-Encoder (로컬)
pip install langchain-cohere        # Cohere Rerank (API)
```

#### 기술 문서: 왜 2단 검색이 필요한가

밀집 검색(Bi-Encoder)은 `sim(embed(q), embed(d))`로 순위를 매긴다.  
질문과 문서를 **독립적으로** 인코딩하기 때문에 상호작용(단어 정렬·부정·세부 조건)을 깊게 모델링하기 어렵다.

Cross-Encoder는 `[CLS] q [SEP] d [SEP]`처럼 **쌍을 함께** 인코딩하고 관련성 점수를 직접 예측한다.  
전체 코퍼스에 쓰면 $O(N)$으로 너무 비싸므로, 실무에서는:

```text
질문 → Bi-Encoder ANN top-k (넓게) → Reranker top-n (정밀) → LLM 답변
```

HyDE·Multi-Query가 **후보를 더 잘 모으는** 기법이라면, Reranking은 **모인 후보의 순서를 고치는** 기법이다.


In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
# OPENAI_API_KEY 필수, COHERE_API_KEY는 Part B에서만 필요
# 경로가 다르면 본인 환경에 맞게 수정한다.
load_dotenv("C:/env/.env")


True

### [0] 공통 준비: LLM, 임베딩, 헬퍼

#### 기술 문서: RAG + Rerank 공통 구성요소

| 구성요소 | 역할 | 본 노트북 선택 |
|---|---|---|
| **LLM** | 최종 grounded 답변 | `gpt-4o-mini`, `temperature=0` |
| **Embeddings** | 1차 벡터 검색 | `text-embedding-3-small` |
| **base k** | 리랭크 전 후보 수 | 넓게 (예: 8) |
| **top_n** | 리랭크 후 최종 수 | 좁게 (예: 3) |

`base k >> top_n` 이어야 리랭커가 순서를 고칠 여지가 생긴다.  
`k == top_n`이면 "재정렬만" 되고 후보 풀은 그대로다.

#### 기술 문서: 설계 포인트

- **`temperature=0`**: 리랭크 전후 비교가 목적이므로 출력을 결정적으로 유지한다.
- **인덱스/질의 임베딩 모델은 동일해야** 벡터 공간이 일치한다.
- **grounded generation**: 시스템 프롬프트로 "문맥 밖의 지식 사용"을 막아 환각을 줄인다.


In [2]:
from typing import List, Optional

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# temperature=0: Naive/Rerank 비교 실험을 재현 가능하게 만든다.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 인덱스와 질의는 반드시 같은 임베딩 모델을 써야 한다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def format_docs(documents: List[Document], show_score: bool = True) -> str:
    """Document 목록을 사람이 읽기 쉬운 문자열로 변환한다.

    리랭커는 보통 relevance_score를 metadata에 넣는다.
    점수가 있으면 함께 출력해 Naive vs Rerank 순위 변화를 확인한다.
    """
    lines = []
    for i, d in enumerate(documents):
        # relevance_score는 아래에서 따로 출력하므로 메타데이터 요약에서는 제외한다.
        meta = {k: v for k, v in d.metadata.items() if k != "relevance_score"}
        meta_str = ", ".join(f"{k}={v}" for k, v in meta.items())
        score = d.metadata.get("relevance_score")
        score_str = f" score={score:.4f}" if show_score and score is not None else ""
        lines.append(f"[{i+1}]{score_str} ({meta_str}) {d.page_content}")
    return "\n\n".join(lines)


def topics(documents: List[Document]) -> List[str]:
    # 비교표용: 본문 대신 topic 라벨만 추출
    return [d.metadata.get("topic", "-") for d in documents]


# grounded generation: 검색 문맥 밖의 지식을 쓰지 않도록 제한한다.
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 문맥만 근거로 질문에 답하라. 문맥에 없으면 모른다고 말하라.\n\n[문맥]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
# LCEL: Prompt → LLM → 문자열 파서
answer_chain = answer_prompt | llm | StrOutputParser()

print("LLM / Embeddings / 헬퍼 준비 완료")


c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


LLM / Embeddings / 헬퍼 준비 완료


### [1] 샘플 코퍼스 & 벡터스토어 (FAISS)

리랭크 효과를 보기 위해 **키워드는 겹치지만 답이 아닌 distractor** 를 섞는다.  
예: "검색 품질"이라는 표현이 청킹·임베딩·평가 문서에도 등장하지만,  
"1차 후보를 다시 점수 매겨 순서를 바꾼다"의 정답은 `reranker`다.

#### 기술 문서: 후보 풀 설계

- **base k=8**: 코퍼스(12개)에서 넓게 가져와 리랭커 입력으로 넘긴다.
- **top_n=3**: LLM 문맥에 넣을 최종 문서 수.
- 실무에서는 보통 `k=20~100`, `top_n=3~10` 범위를 튜닝한다.

#### 기술 문서: 메타데이터 라벨

| 필드 | 용도 |
|---|---|
| `topic` | 결과 비교용 주제 라벨 (검색 알고리즘에는 미사용) |
| `difficulty` | `core` / `related` / `distractor` — 리랭크 전후 품질을 사람이 검증하기 위한 관찰용 태그 |


In [4]:
from langchain_community.vectorstores import FAISS

# metadata.difficulty: core(정답) / related(관련) / distractor(방해 문서)
docs = [
    Document(
        page_content=(
            "리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 "
            "재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 "
            "정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다."
        ),
        metadata={"topic": "reranker", "difficulty": "core"},
    ),
    Document(
        page_content=(
            "교차 인코더(Cross-Encoder)는 질문과 문서를 하나의 입력으로 묶어 "
            "관련성 점수를 직접 예측한다. Bi-Encoder보다 느리지만 쌍 단위 상호작용을 "
            "모델링하므로 재순위화에 적합하다."
        ),
        metadata={"topic": "cross_encoder", "difficulty": "core"},
    ),
    Document(
        page_content=(
            "하이브리드 검색은 BM25 같은 키워드 검색과 벡터 검색을 결합한다. "
            "정확한 용어 매칭과 의미적 유사성을 동시에 활용하며, 점수 정규화 후 "
            "가중 합산하거나 Reciprocal Rank Fusion(RRF)으로 융합한다."
        ),
        metadata={"topic": "hybrid_search", "difficulty": "related"},
    ),
    Document(
        page_content=(
            "쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓힌다. "
            "Multi-Query는 관점을 늘리고, HyDE는 가상 답변 문서를 만들어 "
            "질문-문서 표현 간극을 줄인다. 재현율 향상에 초점을 둔다."
        ),
        metadata={"topic": "query_expansion", "difficulty": "related"},
    ),
    Document(
        page_content=(
            "청킹은 긴 문서를 검색·임베딩에 맞는 크기로 나눈다. "
            "청크가 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 "
            "검색 품질이 떨어진다."
        ),
        metadata={"topic": "chunking", "difficulty": "distractor"},
    ),
    Document(
        page_content=(
            "임베딩 모델은 텍스트를 고차원 벡터로 변환한다. "
            "도메인 특화 문서에서는 도메인 적응형 임베딩이 검색 품질을 개선할 수 있다. "
            "인덱스와 질의는 같은 임베딩 모델을 써야 한다."
        ),
        metadata={"topic": "embedding", "difficulty": "distractor"},
    ),
    Document(
        page_content=(
            "RAGAS는 faithfulness, answer relevancy, context precision/recall 등으로 "
            "RAG 파이프라인의 검색·생성 품질을 정량 평가한다. "
            "리랭크 전후 context precision 변화를 측정하는 데 유용하다."
        ),
        metadata={"topic": "evaluation", "difficulty": "related"},
    ),
    Document(
        page_content=(
            "MMR(Maximal Marginal Relevance)은 관련성과 다양성을 동시에 고려해 "
            "중복이 많은 상위 결과를 줄인다. 리랭커와 목표는 다르지만, "
            "최종 문맥 구성 단계에서 함께 쓰이기도 한다."
        ),
        metadata={"topic": "mmr", "difficulty": "distractor"},
    ),
    Document(
        page_content=(
            "프롬프트 엔지니어링은 LLM 입력을 설계해 답변 품질을 높이는 기법이다. "
            "시스템 역할, few-shot, 출력 형식 제약이 대표적이다. "
            "검색 순위 자체를 바꾸지는 않는다."
        ),
        metadata={"topic": "prompting", "difficulty": "distractor"},
    ),
    Document(
        page_content=(
            "벡터 데이터베이스는 임베딩을 저장하고 ANN 검색을 제공한다. "
            "FAISS, Chroma, Pinecone, pgvector 등이 있다. "
            "1차 후보 검색 속도와 확장성에 영향을 준다."
        ),
        metadata={"topic": "vector_db", "difficulty": "distractor"},
    ),
    Document(
        page_content=(
            "토큰 예산 관리는 검색 문맥을 LLM 컨텍스트 창에 맞게 제한하는 일이다. "
            "상위 문서를 무한정 넣으면 비용과 노이즈가 커지므로, "
            "리랭크 후 top-n만 넣는 전략이 흔하다."
        ),
        metadata={"topic": "token_budget", "difficulty": "related"},
    ),
    Document(
        page_content=(
            "캐싱은 동일·유사 질의의 임베딩·검색·생성 결과를 재사용해 "
            "지연과 비용을 줄인다. 리랭크 API 호출 비용이 클 때 "
            "질의 해시 기반 캐시가 특히 유용하다."
        ),
        metadata={"topic": "caching", "difficulty": "distractor"},
    ),
]

vectorstore = FAISS.from_documents(docs, embeddings)

# 리랭크 전: 넓게(BASE_K) / 리랭크 후: 좁게(TOP_N)
BASE_K = 8
TOP_N = 3
base_retriever = vectorstore.as_retriever(search_kwargs={"k": BASE_K})

print(f"문서 수: {len(docs)}")
print(f"base k={BASE_K}, rerank top_n={TOP_N}")


문서 수: 12
base k=8, rerank top_n=3


### [2] 베이스라인: Naive 벡터 검색

질문을 그대로 임베딩해 top-$k$를 가져온다.  
"검색 품질" 같은 공통 표현 때문에 distractor가 상위에 섞일 수 있다.

#### 기술 문서: Bi-Encoder 실패 패턴

- **표면 유사**: 같은 키워드를 쓰지만 의가 다른 문서가 높은 점수
- **세부 조건 무시**: "다시 점수 매겨 순서를 바꾼다"는 상호작용을 약하게 포착
- **상위권 오염**: 정답이 후보에는 있어도 3~5위에 밀리면 LLM 문맥에서 밀려날 수 있음

이후 Cross-Encoder·Cohere 결과와 비교할 **기준선(baseline)** 이다.


In [5]:
query = "1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은 뭐야?"

# 리랭크 없는 1차 벡터 검색 (베이스라인)
naive_docs = base_retriever.invoke(query)

print("=== [Naive] 검색 결과 (base k) ===")
print(format_docs(naive_docs, show_score=False))
print("\ntopics:", topics(naive_docs))
print("\n=== [Naive] top-3만 문맥으로 답변 ===")
# 벡터 유사도 순서 그대로 앞에서 TOP_N개만 문맥으로 사용
print(
    answer_chain.invoke(
        {"context": format_docs(naive_docs[:TOP_N], show_score=False), "question": query}
    )
)


=== [Naive] 검색 결과 (base k) ===
[1] (topic=reranker, difficulty=core) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다.

[2] (topic=chunking, difficulty=distractor) 청킹은 긴 문서를 검색·임베딩에 맞는 크기로 나눈다. 청크가 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 검색 품질이 떨어진다.

[3] (topic=token_budget, difficulty=related) 토큰 예산 관리는 검색 문맥을 LLM 컨텍스트 창에 맞게 제한하는 일이다. 상위 문서를 무한정 넣으면 비용과 노이즈가 커지므로, 리랭크 후 top-n만 넣는 전략이 흔하다.

[4] (topic=query_expansion, difficulty=related) 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓힌다. Multi-Query는 관점을 늘리고, HyDE는 가상 답변 문서를 만들어 질문-문서 표현 간극을 줄인다. 재현율 향상에 초점을 둔다.

[5] (topic=cross_encoder, difficulty=core) 교차 인코더(Cross-Encoder)는 질문과 문서를 하나의 입력으로 묶어 관련성 점수를 직접 예측한다. Bi-Encoder보다 느리지만 쌍 단위 상호작용을 모델링하므로 재순위화에 적합하다.

[6] (topic=embedding, difficulty=distractor) 임베딩 모델은 텍스트를 고차원 벡터로 변환한다. 도메인 특화 문서에서는 도메인 적응형 임베딩이 검색 품질을 개선할 수 있다. 인덱스와 질의는 같은 임베딩 모델을 써야 한다.

[7] (topic=mmr, difficulty=distractor) MMR(Maximal Marginal Relevanc

---
## Part A. Cross-Encoder Reranker (로컬)

Hugging Face Cross-Encoder를 로컬에서 실행한다. API 키가 없어도 되고, 데이터가 외부로 나가지 않는다.

본 예제는 다국어에 강한 `BAAI/bge-reranker-v2-m3`를 쓴다.  
영문만·경량이면 `cross-encoder/ms-marco-MiniLM-L6-v2`도 가능하다.

### 흐름
1. Bi-Encoder로 top-$k$ 후보 검색
2. 각 (질문, 문서) 쌍에 Cross-Encoder 점수 부여
3. 점수 내림차순 정렬 후 top-$n$ 반환
4. (선택) 최종 문서로 답변 생성

#### 기술 문서: Cross-Encoder 점수

모델마다 점수 스케일이 다르다 (로짓, 0~1 확률 등).  
**절대값보다 상대 순위**가 중요하다. 임계값 필터를 두려면 코퍼스에서 점수 분포를 먼저 확인한다.

#### 기술 문서: 모델 선택

- 다국어(한국어 포함): `BAAI/bge-reranker-v2-m3`
- 영문·경량: `cross-encoder/ms-marco-MiniLM-L6-v2`
- 첫 실행은 Hugging Face에서 모델을 받느라 느릴 수 있다 (이후 캐시 사용).


### [A-1] 수동 구현: Cross-Encoder로 재점수화

LangChain 없이 `sentence_transformers.CrossEncoder`만으로 동작을 확인한다.

#### 기술 문서: 핵심 API

- **`pairs`**: `(질문, 문서 본문)` 튜플 리스트 — Cross-Encoder 입력 단위
- **`predict()`**: 각 쌍의 관련성 점수 배열 반환
- **`deepcopy()`**: 원본 `Document`/`metadata`를 건드리지 않고 점수를 기록하기 위해 사용
- **`metadata["relevance_score"]`**: 리랭크 점수 저장 관례 키


In [6]:
from copy import deepcopy

from sentence_transformers import CrossEncoder

# 첫 실행 시 Hugging Face에서 모델을 다운로드한다.
# CPU에서도 동작하지만, 후보 수가 많으면 GPU가 유리하다.
ce_model_name = "BAAI/bge-reranker-v2-m3"
cross_encoder = CrossEncoder(ce_model_name)


def rerank_with_cross_encoder(
    question: str,
    documents: List[Document],
    model: CrossEncoder,
    top_n: int = 3,
) -> List[Document]:
    """(질문, 문서) 쌍 점수로 재정렬한 뒤 top_n을 반환한다."""
    pairs = [(question, d.page_content) for d in documents]
    scores = model.predict(pairs)

    ranked = []
    for doc, score in sorted(
        zip(documents, scores), key=lambda x: float(x[1]), reverse=True
    ):
        # 원본 Document를 오염시키지 않도록 복사 후 점수 기록
        new_doc = deepcopy(doc)
        new_doc.metadata = dict(doc.metadata)
        new_doc.metadata["relevance_score"] = float(score)
        ranked.append(new_doc)
    return ranked[:top_n]


# 주의: 리랭커는 1차 후보(naive_docs) 밖의 문서를 새로 찾아오지 못한다.
ce_manual_docs = rerank_with_cross_encoder(
    query, naive_docs, cross_encoder, top_n=TOP_N
)

print("=== [Cross-Encoder 수동] 리랭크 결과 ===")
print(format_docs(ce_manual_docs))
print("\nNaive topics :", topics(naive_docs))
print("CE topics    :", topics(ce_manual_docs))
print("\n=== [Cross-Encoder 수동] 답변 ===")
print(
    answer_chain.invoke(
        {"context": format_docs(ce_manual_docs), "question": query}
    )
)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

=== [Cross-Encoder 수동] 리랭크 결과 ===
[1] score=0.9173 (topic=reranker, difficulty=core) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다.

[2] score=0.0210 (topic=cross_encoder, difficulty=core) 교차 인코더(Cross-Encoder)는 질문과 문서를 하나의 입력으로 묶어 관련성 점수를 직접 예측한다. Bi-Encoder보다 느리지만 쌍 단위 상호작용을 모델링하므로 재순위화에 적합하다.

[3] score=0.0044 (topic=prompting, difficulty=distractor) 프롬프트 엔지니어링은 LLM 입력을 설계해 답변 품질을 높이는 기법이다. 시스템 역할, few-shot, 출력 형식 제약이 대표적이다. 검색 순위 자체를 바꾸지는 않는다.

Naive topics : ['reranker', 'chunking', 'token_budget', 'query_expansion', 'cross_encoder', 'embedding', 'mmr', 'prompting']
CE topics    : ['reranker', 'cross_encoder', 'prompting']

=== [Cross-Encoder 수동] 답변 ===
리랭커(Reranker)입니다.


In [7]:
# cross-encoder/ms-marco-MiniLM-L6-v2 사용 예제
from sentence_transformers import CrossEncoder

# Cross-Encoder 모델
model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

query = "RAG란 무엇인가?"

documents = [
    "RAG는 검색 증강 생성 기술이다.",
    "GPU는 병렬 연산 장치이다.",
    "LangChain은 LLM 프레임워크이다."
]

# 질문-문서 쌍 생성
pairs = [[query, doc] for doc in documents]

# 관련성 점수 계산
scores = model.predict(pairs)

for score, doc in zip(scores, documents):
    print(f"{score:.3f} : {doc}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

7.591 : RAG는 검색 증강 생성 기술이다.
1.859 : GPU는 병렬 연산 장치이다.
2.467 : LangChain은 LLM 프레임워크이다.


### [A-2] LangChain `CrossEncoderReranker` + `ContextualCompressionRetriever`

LangChain v1에서는 retriever/compressor가 `langchain-classic`에 있다.

```python
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
```

#### 기술 문서: ContextualCompressionRetriever

```text
질문
  │
  ▼
base_retriever.invoke(질문)     → Document[k]
  │
  ▼
base_compressor.compress_documents(docs, 질문)
  │  CrossEncoderReranker: 재점수화 + top_n 슬라이스
  ▼
Document[top_n]  (+ metadata.relevance_score)
```

`ContextualCompressionRetriever`는 리랭크뿐 아니라 LLM 기반 압축·필터에도 쓰는 공통 래퍼다.  
인터페이스만 맞으면 compressor를 교체해 A/B 실험하기 쉽다.

| 수동 구현 | LangChain 래퍼 |
|---|---|
| `CrossEncoder(...)` | `HuggingFaceCrossEncoder(...)` |
| `rerank_with_cross_encoder(...)` | `CrossEncoderReranker` + `ContextualCompressionRetriever` |


In [8]:
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

hf_ce = HuggingFaceCrossEncoder(model_name=ce_model_name)
ce_compressor = CrossEncoderReranker(model=hf_ce, top_n=TOP_N)

# base_retriever(검색) → ce_compressor(리랭크)를 한 번에 연결
ce_retriever = ContextualCompressionRetriever(
    base_compressor=ce_compressor,
    base_retriever=base_retriever,
)

ce_docs = ce_retriever.invoke(query)

print("=== [LangChain CrossEncoderReranker] 결과 ===")
print(format_docs(ce_docs))
print("\ntopics:", topics(ce_docs))
print("\n=== 답변 ===")
print(
    answer_chain.invoke({"context": format_docs(ce_docs), "question": query})
)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

=== [LangChain CrossEncoderReranker] 결과 ===
[1] (topic=evaluation, difficulty=related) RAGAS는 faithfulness, answer relevancy, context precision/recall 등으로 RAG 파이프라인의 검색·생성 품질을 정량 평가한다. 리랭크 전후 context precision 변화를 측정하는 데 유용하다.

[2] (topic=reranker, difficulty=core) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다.

[3] (topic=token_budget, difficulty=related) 토큰 예산 관리는 검색 문맥을 LLM 컨텍스트 창에 맞게 제한하는 일이다. 상위 문서를 무한정 넣으면 비용과 노이즈가 커지므로, 리랭크 후 top-n만 넣는 전략이 흔하다.

topics: ['evaluation', 'reranker', 'token_budget']

=== 답변 ===
문맥에 RAG에 대한 정의나 설명이 포함되어 있지 않으므로, RAG가 무엇인지 모릅니다.


---
## Part B. Cohere Rerank (API)

Cohere Rerank는 관리형 API다. 모델 로딩·인프라 없이 높은 품질의 재순위화를 쓸 수 있다.

```text
pip install langchain-cohere
```

환경변수 `COHERE_API_KEY`가 필요하다. 없으면 아래 셀은 건너뛴다.

| 모델 예 | 용도 |
|---|---|
| `rerank-multilingual-v3.0` | 한국어·다국어 |
| `rerank-english-v3.0` | 영문 |
| `rerank-v3.5` | 최신 통합 모델 (계정/문서 기준 확인) |

#### 기술 문서: 로컬 CE vs Cohere

| | Cross-Encoder (로컬) | Cohere Rerank |
|---|---|---|
| 데이터 주권 | 온프레미스 가능 | 텍스트가 API로 전송 |
| 운영 | GPU/메모리·버전 관리 | API 키·쿼터·단가 |
| 지연 | 모델·하드웨어 의존 | 네트워크 + API |
| 교체 비용 | 모델 파일 교체 | `model=` 문자열 변경 |

`ContextualCompressionRetriever` 구조는 Part A와 동일하고, compressor만 `CohereRerank`로 교체한다.


In [8]:
# ! pip install langchain-cohere

In [9]:
cohere_key = os.getenv("COHERE_API_KEY")

if not cohere_key:
    print("COHERE_API_KEY가 없어 Cohere Rerank 셀을 건너뜁니다.")
    print(".env에 COHERE_API_KEY를 추가한 뒤 이 셀을 다시 실행하세요.")
    cohere_docs = None
else:
    from langchain_cohere import CohereRerank

    # 한국어 코퍼스이므로 multilingual 모델을 사용
    cohere_compressor = CohereRerank(
        model="rerank-multilingual-v3.0",
        top_n=TOP_N,
    )
    # Part A와 동일 패턴 — compressor만 API로 교체
    cohere_retriever = ContextualCompressionRetriever(
        base_compressor=cohere_compressor,
        base_retriever=base_retriever,
    )

    cohere_docs = cohere_retriever.invoke(query)

    print("=== [Cohere Rerank] 결과 ===")
    print(format_docs(cohere_docs))
    print("\ntopics:", topics(cohere_docs))
    print("\n=== 답변 ===")
    print(
        answer_chain.invoke(
            {"context": format_docs(cohere_docs), "question": query}
        )
    )


=== [Cohere Rerank] 결과 ===
[1] score=0.5968 (topic=evaluation, difficulty=related) RAGAS는 faithfulness, answer relevancy, context precision/recall 등으로 RAG 파이프라인의 검색·생성 품질을 정량 평가한다. 리랭크 전후 context precision 변화를 측정하는 데 유용하다.

[2] score=0.0448 (topic=reranker, difficulty=core) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지하면서 정밀도를 높이는 데 효과적이다. Cohere Rerank, bge-reranker 등이 대표적이다.

[3] score=0.0306 (topic=prompting, difficulty=distractor) 프롬프트 엔지니어링은 LLM 입력을 설계해 답변 품질을 높이는 기법이다. 시스템 역할, few-shot, 출력 형식 제약이 대표적이다. 검색 순위 자체를 바꾸지는 않는다.

topics: ['evaluation', 'reranker', 'prompting']

=== 답변 ===
문맥에 RAG에 대한 구체적인 정의는 없지만, RAGAS는 RAG 파이프라인의 검색·생성 품질을 정량 평가하는 방법으로 언급되고 있습니다. 따라서 RAG는 검색과 생성의 품질을 평가하는 시스템일 가능성이 있습니다.


### [B-1] (선택) Cohere SDK로 직접 호출

LangChain 없이 Cohere 클라이언트만으로도 같은 작업을 할 수 있다.  
디버깅·벤치마크 시 원시 응답을 보기 좋다.

#### 기술 문서: Cohere 원시 응답 구조

| 필드 | 의미 |
|---|---|
| `response.results` | 점수순으로 정렬된 결과 리스트 |
| `item.index` | 입력 `documents` 리스트에서의 **원래 위치** (순위가 아님) |
| `item.relevance_score` | 관련성 점수 (모델·버전별 스케일 상이 가능) |

LangChain `CohereRerank`는 이 `index`로 원본 `Document`를 매핑해 `metadata`에 점수를 채운다.


In [10]:
if not cohere_key:
    print("COHERE_API_KEY 없음 — 건너뜀")
else:
    import cohere

    client = cohere.Client(api_key=cohere_key)
    # API는 문자열 리스트를 받음. item.index는 이 리스트 기준 위치다.
    candidates = [d.page_content for d in naive_docs]

    response = client.rerank(
        model="rerank-multilingual-v3.0",
        query=query,
        documents=candidates,
        top_n=TOP_N,
    )

    print("=== [Cohere SDK] 원시 리랭크 ===")
    for i, item in enumerate(response.results, start=1):
        src = naive_docs[item.index]
        print(
            f"[{i}] score={item.relevance_score:.4f} "
            f"(topic={src.metadata.get('topic')}) "
            f"{src.page_content[:80]}..."
        )


=== [Cohere SDK] 원시 리랭크 ===
[1] score=0.1413 (topic=chunking) 청킹은 긴 문서를 검색·임베딩에 맞는 크기로 나눈다. 청크가 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 검색 품질이 떨어진다....
[2] score=0.0448 (topic=reranker) 리랭커(Reranker)는 1차 검색으로 가져온 후보 문서를 교차 인코더 등으로 재점수화하여 상위 결과를 재정렬한다. 검색 단계의 재현율을 유지...
[3] score=0.0306 (topic=prompting) 프롬프트 엔지니어링은 LLM 입력을 설계해 답변 품질을 높이는 기법이다. 시스템 역할, few-shot, 출력 형식 제약이 대표적이다. 검색 순...


---
### [3] Naive vs Cross-Encoder vs Cohere 비교

여러 질문에 대해 상위 topic 순서를 나란히 비교한다.

#### 기술 문서: 비교표 읽는 법

- 같은 질문에서 **topic 리스트 순서** = 각 방식이 판단한 관련성 순위
- Naive와 Cross-Encoder 순위가 다르면 리랭커가 실제로 순서를 바꾼 것
- `(skipped)`는 `COHERE_API_KEY` 없음 (오류 아님)
- 소규모 코퍼스에서는 세 방식 차이가 작을 수 있다


In [11]:
from unicodedata import east_asian_width

test_queries = [
    "1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?",  # → reranker
    "질문과 문서를 한 쌍으로 넣어 관련성 점수를 직접 내는 모델은?",  # → cross_encoder
    "키워드 검색이랑 벡터 검색을 같이 쓰는 방법은?",  # → hybrid_search
    "짧은 질문을 여러 표현으로 바꿔 검색 범위를 넓히는 방법은?",  # → query_expansion
]


def retrieve_naive(q: str) -> List[Document]:
    return base_retriever.invoke(q)[:TOP_N]


def retrieve_ce(q: str) -> List[Document]:
    return ce_retriever.invoke(q)


def retrieve_cohere(q: str) -> Optional[List[Document]]:
    if not cohere_key:
        return None
    return cohere_retriever.invoke(q)


def disp_width(s: str) -> int:
    # 한글 등 전각 문자는 터미널에서 폭 2로 보이므로, 정렬 시 이를 반영한다.
    return sum(2 if east_asian_width(ch) in ("F", "W") else 1 for ch in s)


def pad(s: str, width: int) -> str:
    return s + " " * max(0, width - disp_width(s))


def fmt_topics(t) -> str:
    # 리스트 표기(['a', 'b']) 대신 순위 화살표로 짧게 표시해 열 폭을 맞춘다.
    if isinstance(t, str):
        return t
    return " > ".join(t)


# 표시 폭 기준 열 너비 (한글 질문 열은 실제 글자 수보다 넓게)
COL_Q, COL_N, COL_CE, COL_CO = 62, 44, 44, 40

header = (
    f"{pad('질문', COL_Q)} | {pad('Naive', COL_N)} | "
    f"{pad('Cross-Encoder', COL_CE)} | {pad('Cohere', COL_CO)}"
)
print(header)
print("-" * disp_width(header))

for q in test_queries:
    n_topics = topics(retrieve_naive(q))
    c_topics = topics(retrieve_ce(q))
    co = retrieve_cohere(q)
    co_topics = topics(co) if co is not None else "(skipped)"

    print(
        f"{pad(q, COL_Q)} | {pad(fmt_topics(n_topics), COL_N)} | "
        f"{pad(fmt_topics(c_topics), COL_CE)} | {pad(fmt_topics(co_topics), COL_CO)}"
    )


질문                                                           | Naive                                        | Cross-Encoder                                | Cohere                                  
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?       | reranker > chunking > token_budget           | reranker > cross_encoder > token_budget      | reranker > cross_encoder > prompting    
질문과 문서를 한 쌍으로 넣어 관련성 점수를 직접 내는 모델은?   | embedding > query_expansion > cross_encoder  | cross_encoder > query_expansion > embedding  | cross_encoder > embedding > reranker    
키워드 검색이랑 벡터 검색을 같이 쓰는 방법은?                  | hybrid_search > vector_db > reranker         | hybrid_search > vector_db > embedding        | hybrid_search > embedding > vector_db   
짧은 질문을 여러 표현으로 바꿔 검색 범위를 넓히는 방법은?      | query_expansion > reranker >

### [4] 하이퍼파라미터 감각: k와 top_n

`BASE_K`를 바꿔 보면, 후보가 너무 적으면 정답이 풀 밖으로 나가고,  
너무 많으면 리랭크 비용만 커진다.

#### 기술 문서: k / top_n 튜닝

- **k**: 정답이 후보에 들어올 만큼 충분히 넓게
- **top_n**: LLM 문맥이 깔끔하도록 좁게
- **관찰 패턴**
  - k가 너무 작음 → 정답이 후보 밖 → 리랭크해도 못 올림
  - k를 키움 → 정답이 들어와 상위권이 안정
  - k를 더 키워도 top_n이 안 바뀌면 충분 (비용만 증가)


In [12]:
probe_query = "1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?"

print(f"질문: {probe_query}\n")
# k를 늘려가며 후보 풀·리랭크 결과 변화를 관찰
for k in [3, 5, 8, 12]:
    candidates = vectorstore.similarity_search(probe_query, k=k)
    reranked = rerank_with_cross_encoder(
        probe_query, candidates, cross_encoder, top_n=TOP_N
    )
    print(
        f"k={k:<2} → 후보 topics={topics(candidates)} "
        f"| rerank top-{TOP_N}={topics(reranked)}"
    )


질문: 1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?

k=3  → 후보 topics=['reranker', 'chunking', 'token_budget'] | rerank top-3=['reranker', 'token_budget', 'chunking']
k=5  → 후보 topics=['reranker', 'chunking', 'token_budget', 'query_expansion', 'cross_encoder'] | rerank top-3=['reranker', 'cross_encoder', 'token_budget']
k=8  → 후보 topics=['reranker', 'chunking', 'token_budget', 'query_expansion', 'cross_encoder', 'embedding', 'mmr', 'prompting'] | rerank top-3=['reranker', 'cross_encoder', 'token_budget']
k=12 → 후보 topics=['reranker', 'chunking', 'token_budget', 'query_expansion', 'cross_encoder', 'embedding', 'mmr', 'prompting', 'hybrid_search', 'vector_db', 'evaluation', 'caching'] | rerank top-3=['reranker', 'cross_encoder', 'token_budget']


---
### [정리] Reranking 체크리스트

| 구분 | Cross-Encoder (로컬) | Cohere Rerank |
|---|---|---|
| 구현 | `CrossEncoder` / `CrossEncoderReranker` | `CohereRerank` |
| 래퍼 | `ContextualCompressionRetriever` | 동일 |
| 강점 | 데이터 비공개, 무제한 호출(하드웨어 한도) | 운영 단순, 품질·다국어 |
| 약점 | 모델 다운로드·추론 자원 | API 비용·외부 전송 |

**구현 포인트 (LangChain v1)**
- `from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever`
- `from langchain_classic.retrievers.document_compressors import CrossEncoderReranker`
- `from langchain_community.cross_encoders import HuggingFaceCrossEncoder`
- `from langchain_cohere import CohereRerank`

**언제 쓰면 좋은가**
- 1차 검색 재현율은 괜찮은데 상위 정밀도가 부족할 때
- Multi-Query / HyDE로 후보가 늘어난 뒤 순서를 정리할 때
- LLM 컨텍스트에 넣을 문서를 3~5개로 줄여야 할 때

#### 기술 문서: 실무 조합 패턴

```text
사용자 질문
    │
    ├─ (선택) Multi-Query / HyDE / Self-Query  → recall↑
    │
    ▼
Bi-Encoder / Hybrid 검색  (k=20~100)
    │
    ▼
Reranker  (Cross-Encoder 또는 Cohere, top_n=3~10)  → precision↑
    │
    ▼
LLM grounded 답변
```

리랭커는 **없는 정답을 만들어 내지 않는다**.  
정답이 1차 후보에 없으면 리랭크해도 못 올린다. 그래서 recall 전략(넓은 k, Multi-Query, Hybrid)과 precision 전략(Rerank)을 함께 설계한다.
